# 04b — CART: Classification and Regression Trees
**Referencias:** ESL Cap. 9.2 (Tree-Based Methods) · Géron Cap. 6

## ¿Qué es CART? (ESL 9.2)
CART particiona el espacio de features de forma **binaria y recursiva**, minimizando una función de impureza en cada split.
El árbol resultante es una función constante a trozos:
$$f(x) = \sum_{m=1}^{M} c_m \cdot \mathbf{1}(x \in R_m)$$
donde $R_m$ son las regiones terminales (hojas) y $c_m$ la predicción en cada hoja.

## Criterio de split — Clasificación (ESL 9.2.3)
En cada nodo se busca el split $(j, t)$ — feature $j$, umbral $t$ — que minimiza la impureza ponderada de los hijos.

| Criterio | Fórmula | Nota |
|---|---|---|
| **Gini** | $\sum_k \hat{p}_{mk}(1-\hat{p}_{mk})$ | Default en sklearn, más rápido |
| **Entropy** | $-\sum_k \hat{p}_{mk}\log\hat{p}_{mk}$ | Information Gain — equivalente en la práctica |
| **Log-loss** | igual que entropy, con corrección | Desde sklearn 1.1 |

## Criterio de split — Regresión (ESL 9.2.1)
$$\min_{j,t} \left[ \sum_{x_i \in R_1(j,t)} (y_i - \bar{y}_{R_1})^2 + \sum_{x_i \in R_2(j,t)} (y_i - \bar{y}_{R_2})^2 \right]$$

## Poda — Cost-Complexity Pruning (ESL 9.2.2)
Un árbol completo sobreajusta. La poda con penalización de complejidad resuelve:
$$\min_T \sum_{m=1}^{|T|} \sum_{x_i \in R_m} (y_i - \hat{y}_{R_m})^2 + \alpha |T|$$
donde $|T|$ es el número de hojas y $\alpha \geq 0$ controla el trade-off sesgo-varianza.
En sklearn: `ccp_alpha` — mayor valor → árbol más pequeño.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, validation_curve
from sklearn.tree import (
    DecisionTreeClassifier, DecisionTreeRegressor,
    plot_tree, export_text
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    mean_squared_error, r2_score
)

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# Dataset: comportamiento de usuarios en plataforma SaaS
n = 2000
sessions       = np.random.randint(1, 25, n)
time_on_site   = np.random.uniform(30, 800, n)
pages          = np.random.uniform(1, 10, n)
days_since_reg = np.random.randint(0, 30, n)
features_used  = np.random.randint(0, 10, n)
channel        = np.random.choice(['organic', 'paid', 'email', 'direct'], n)
device         = np.random.choice(['mobile', 'desktop', 'tablet'], n)
plan           = np.random.choice(['free', 'trial', 'pro'], n, p=[0.6, 0.25, 0.15])

logit = (
    -3.5 + sessions * 0.12 + time_on_site * 0.002 + pages * 0.18
    - days_since_reg * 0.06 + features_used * 0.25
    + np.where(channel == 'email', 0.9, 0)
    + np.where(device == 'desktop', 0.4, 0)
    + np.where(plan == 'trial', 1.2, np.where(plan == 'pro', 2.0, 0))
)
prob = 1 / (1 + np.exp(-logit))
converted = (np.random.uniform(0, 1, n) < prob).astype(int)

revenue = (
    100 + sessions * 15 + time_on_site * 0.25 + pages * 10
    + features_used * 20 - days_since_reg * 3
    + np.where(plan == 'trial', 80, np.where(plan == 'pro', 300, 0))
    + np.where(device == 'desktop', 40, 0)
    + np.random.normal(0, 60, n)
).clip(0)

df = pd.DataFrame({
    'sessions': sessions, 'time_on_site': time_on_site.round(0),
    'pages': pages.round(2), 'days_since_reg': days_since_reg,
    'features_used': features_used, 'channel': channel,
    'device': device, 'plan': plan,
    'converted': converted, 'revenue': revenue.round(2),
})

num_features = ['sessions', 'time_on_site', 'pages', 'days_since_reg', 'features_used']
cat_features = ['channel', 'device', 'plan']

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
])

X = df[num_features + cat_features]
y_clf = df['converted']
y_reg = df['revenue']

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_clf, y_reg, test_size=0.2, random_state=42, stratify=y_clf
)
print(f'Dataset: {n} filas | Tasa conversión: {y_clf.mean():.2%} | Revenue media: ${y_reg.mean():.0f}')

## 1 — Árbol de Clasificación: baseline y visualización (Géron Cap. 6)

In [ ]:
# Árbol sin restricciones vs árbol con max_depth controlado
clf_full  = Pipeline([('prep', preprocessor), ('tree', DecisionTreeClassifier(random_state=42))])
clf_d3    = Pipeline([('prep', preprocessor), ('tree', DecisionTreeClassifier(max_depth=3, random_state=42))])
clf_d5    = Pipeline([('prep', preprocessor), ('tree', DecisionTreeClassifier(max_depth=5, random_state=42))])

for name, pipe in [('Sin restricción', clf_full), ('max_depth=3', clf_d3), ('max_depth=5', clf_d5)]:
    pipe.fit(X_train, yc_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    tree   = pipe.named_steps['tree']
    print(f'{name:<20} depth={tree.get_depth():>2}  leaves={tree.get_n_leaves():>4}  '
          f'Acc={accuracy_score(yc_test, y_pred):.3f}  '
          f'F1={f1_score(yc_test, y_pred):.3f}  '
          f'AUC={roc_auc_score(yc_test, y_prob):.3f}')

print('\n→ El árbol completo tiene más leaves pero no mejor AUC: overfitting clásico.')

In [ ]:
# Visualizar el árbol de profundidad 3
X_prep_vis = preprocessor.fit_transform(X_train)
cat_names_vis = list(preprocessor.named_transformers_['cat'].get_feature_names_out())
feat_names_vis = num_features + cat_names_vis

tree_vis = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_vis.fit(X_prep_vis, yc_train)

fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(
    tree_vis, feature_names=feat_names_vis,
    class_names=['No conv.', 'Convertido'],
    filled=True, rounded=True, fontsize=7.5, ax=ax,
    impurity=True, proportion=False,
)
ax.set_title('CART — Árbol de clasificación (max_depth=3) | Criterio: Gini', pad=12)
plt.tight_layout()
plt.show()

print(export_text(tree_vis, feature_names=feat_names_vis, max_depth=2))

## 2 — Árbol de Regresión (ESL 9.2.1)

In [ ]:
for name, depth in [('Sin restricción', None), ('max_depth=3', 3), ('max_depth=5', 5), ('max_depth=8', 8)]:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('tree', DecisionTreeRegressor(max_depth=depth, random_state=42))
    ])
    pipe.fit(X_train, yr_train)
    y_pred = pipe.predict(X_test)
    rmse   = mean_squared_error(yr_test, y_pred) ** 0.5
    r2     = r2_score(yr_test, y_pred)
    leaves = pipe.named_steps['tree'].get_n_leaves()
    print(f'{name:<20} leaves={leaves:>4}  RMSE=${rmse:6.1f}  R²={r2:.4f}')

print('\n→ Árbol muy profundo memoriza el training set (RMSE train ≈ 0) pero no generaliza.')

## 3 — Feature Importances (ESL 9.2 / Géron Cap. 6)
Sklearn reporta la **reducción total de impureza** (Gini o MSE) que aporta cada feature, promediada y normalizada.
> ⚠️ Las importancias de árboles individuales pueden estar sesgadas hacia features con alta cardinalidad. Usar `permutation_importance` para validar.

In [ ]:
from sklearn.inspection import permutation_importance

X_prep_full = preprocessor.fit_transform(X_train)
X_prep_test = preprocessor.transform(X_test)
cat_names   = list(preprocessor.named_transformers_['cat'].get_feature_names_out())
feat_names  = num_features + cat_names

tree_imp = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_imp.fit(X_prep_full, yc_train)

# Gini importance (built-in)
gini_imp = pd.Series(tree_imp.feature_importances_, index=feat_names).sort_values(ascending=False)

# Permutation importance (más confiable)
perm = permutation_importance(tree_imp, X_prep_test, yc_test, n_repeats=20, random_state=42, scoring='roc_auc')
perm_imp = pd.Series(perm.importances_mean, index=feat_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

top_n = 10
for ax, imp, title, color in [
    (axes[0], gini_imp.head(top_n), 'Gini Importance (built-in)', '#4361ee'),
    (axes[1], perm_imp.head(top_n), 'Permutation Importance (AUC)', '#f72585'),
]:
    ax.barh(imp.index[::-1], imp.values[::-1], color=color, alpha=0.85)
    ax.set_xlabel('Importancia')
    ax.set_title(title)

plt.suptitle('Feature Importances — Árbol de Clasificación (max_depth=5)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 5 Gini:')        ; print(gini_imp.head().to_string())
print('\nTop 5 Permutation:'); print(perm_imp.head().to_string())

## 4 — Hiperparámetros: Curva de Validación (Géron Cap. 6 / ESL 9.2.2)

| Hiperparámetro | Efecto | Trade-off |
|---|---|---|
| `max_depth` | Límite de profundidad | Bajo → underfitting, Alto → overfitting |
| `min_samples_split` | Mínimo de muestras para dividir un nodo | Alto → árbol más pequeño, menos overfitting |
| `min_samples_leaf` | Mínimo de muestras en cada hoja | Alto → suaviza predicciones |
| `max_features` | Features consideradas en cada split | Bajo → más diversidad, pero puede subir sesgo |
| `ccp_alpha` | Penalización de poda post-entrenamiento | Alto → árbol más pequeño |
| `criterion` | `gini` / `entropy` / `log_loss` | Empíricamente similares |

In [ ]:
# Validation curves para max_depth y min_samples_leaf
X_prep_tr = preprocessor.fit_transform(X_train)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

param_configs = [
    ('max_depth',        np.arange(1, 21),     axes[0], 'max_depth'),
    ('min_samples_leaf', np.arange(1, 60, 3),  axes[1], 'min_samples_leaf'),
]

for param_name, param_range, ax, xlabel in param_configs:
    train_scores, test_scores = validation_curve(
        DecisionTreeClassifier(random_state=42),
        X_prep_tr, yc_train,
        param_name=param_name, param_range=param_range,
        cv=5, scoring='roc_auc', n_jobs=-1
    )
    tr_mean, tr_std = train_scores.mean(1), train_scores.std(1)
    te_mean, te_std = test_scores.mean(1),  test_scores.std(1)

    ax.plot(param_range, tr_mean, 'o-', color='#4361ee', linewidth=2, label='Train AUC')
    ax.fill_between(param_range, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color='#4361ee')
    ax.plot(param_range, te_mean, 's-', color='#f72585', linewidth=2, label='CV AUC')
    ax.fill_between(param_range, te_mean - te_std, te_mean + te_std, alpha=0.15, color='#f72585')
    ax.set_xlabel(xlabel); ax.set_ylabel('AUC')
    ax.set_title(f'Validation Curve — {param_name}')
    ax.legend(fontsize=9)
    best = param_range[np.argmax(te_mean)]
    ax.axvline(best, color='#06d6a0', linestyle='--', linewidth=1.5, label=f'Óptimo: {best}')
    ax.legend(fontsize=9)

plt.suptitle('Sesgo-Varianza: Train vs CV — DecisionTreeClassifier', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 5 — GridSearchCV: búsqueda exhaustiva de hiperparámetros

In [ ]:
param_grid = {
    'tree__max_depth':        [3, 5, 7, 10, None],
    'tree__min_samples_leaf': [1, 5, 10, 20],
    'tree__min_samples_split':[2, 10, 20],
    'tree__criterion':        ['gini', 'entropy'],
}

pipe_gs = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=42)),
])

gs = GridSearchCV(
    pipe_gs, param_grid, cv=5,
    scoring='roc_auc', n_jobs=-1, refit=True
)
gs.fit(X_train, yc_train)

print('Mejores hiperparámetros:')
for k, v in gs.best_params_.items():
    print(f'  {k:<30} = {v}')
print(f'\nCV AUC (best): {gs.best_score_:.4f}')

y_pred_gs = gs.predict(X_test)
y_prob_gs = gs.predict_proba(X_test)[:, 1]
print(f'Test  AUC:     {roc_auc_score(yc_test, y_prob_gs):.4f}')
print(f'Test  Acc:     {accuracy_score(yc_test, y_pred_gs):.4f}')
print(f'Test  F1:      {f1_score(yc_test, y_pred_gs):.4f}')

In [ ]:
# Visualizar el espacio de búsqueda: max_depth vs min_samples_leaf (promedios de CV)
results = pd.DataFrame(gs.cv_results_)

pivot_data = results[
    (results['param_tree__criterion'] == 'gini') &
    (results['param_tree__min_samples_split'] == 2)
].copy()

# Reemplazar None por 99 para poder hacer pivot
pivot_data['param_tree__max_depth'] = pivot_data['param_tree__max_depth'].fillna(99)

pivot = pivot_data.pivot_table(
    index='param_tree__max_depth',
    columns='param_tree__min_samples_leaf',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', vmin=pivot.values.min())
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns)
ax.set_yticklabels(['None' if v == 99 else str(int(v)) for v in pivot.index])
ax.set_xlabel('min_samples_leaf'); ax.set_ylabel('max_depth')
ax.set_title('CV AUC — GridSearch (criterion=gini, min_samples_split=2)')
plt.colorbar(im, ax=ax, label='Mean CV AUC')

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.3f}', ha='center', va='center', fontsize=7.5)

plt.tight_layout()
plt.show()

## 6 — Cost-Complexity Pruning con `ccp_alpha` (ESL 9.2.2 / Géron Cap. 6)
sklearn implementa la poda minimal cost-complexity: dado el árbol completo, calcula la secuencia de `ccp_alpha` que produce el mejor árbol podado para cada tamaño. Se elige $\alpha$ por cross-validation.

In [ ]:
X_prep_ccp = preprocessor.fit_transform(X_train)
X_prep_ccp_te = preprocessor.transform(X_test)

# Obtener la secuencia de alphas del árbol completo
full_tree = DecisionTreeClassifier(random_state=42)
path = full_tree.cost_complexity_pruning_path(X_prep_ccp, yc_train)
ccp_alphas = path.ccp_alphas[:-1]  # excluir el último (árbol con 1 hoja)

print(f'Número de alphas en la secuencia: {len(ccp_alphas)}')
print(f'Rango de alphas: [{ccp_alphas.min():.6f}, {ccp_alphas.max():.4f}]')

# Entrenar un árbol por cada alpha
trees, train_aucs, test_aucs, n_leaves = [], [], [], []
for alpha in ccp_alphas:
    t = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    t.fit(X_prep_ccp, yc_train)
    trees.append(t)
    train_aucs.append(roc_auc_score(yc_train, t.predict_proba(X_prep_ccp)[:, 1]))
    test_aucs.append(roc_auc_score(yc_test,  t.predict_proba(X_prep_ccp_te)[:, 1]))
    n_leaves.append(t.get_n_leaves())

best_idx = np.argmax(test_aucs)
print(f'\nAlpha óptimo (test): {ccp_alphas[best_idx]:.6f}')
print(f'Hojas:               {n_leaves[best_idx]}')
print(f'Test AUC:            {test_aucs[best_idx]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(ccp_alphas, train_aucs, 'o-', color='#4361ee', linewidth=1.5, markersize=3, label='Train AUC')
axes[0].plot(ccp_alphas, test_aucs,  's-', color='#f72585', linewidth=1.5, markersize=3, label='Test AUC')
axes[0].axvline(ccp_alphas[best_idx], color='#06d6a0', linestyle='--', linewidth=2, label=f'α*={ccp_alphas[best_idx]:.5f}')
axes[0].set_xlabel('ccp_alpha'); axes[0].set_ylabel('AUC')
axes[0].set_title('AUC vs ccp_alpha'); axes[0].legend(fontsize=9)

axes[1].plot(ccp_alphas, n_leaves, 'o-', color='#7209b7', linewidth=1.5, markersize=3)
axes[1].axvline(ccp_alphas[best_idx], color='#06d6a0', linestyle='--', linewidth=2, label=f'α*={ccp_alphas[best_idx]:.5f}')
axes[1].set_xlabel('ccp_alpha'); axes[1].set_ylabel('N° de hojas')
axes[1].set_title('Tamaño del árbol vs ccp_alpha'); axes[1].legend(fontsize=9)

plt.suptitle('Cost-Complexity Pruning (ESL 9.2.2)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 7 — GridSearchCV sobre ccp_alpha (práctica recomendada)

In [ ]:
# En producción: buscar ccp_alpha con CV junto con otros hiperparámetros
alphas_grid = np.linspace(0, ccp_alphas.max() * 0.5, 20)

param_grid_ccp = {
    'tree__ccp_alpha':        alphas_grid,
    'tree__min_samples_leaf': [1, 5, 10],
}

pipe_ccp = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=42)),
])

gs_ccp = GridSearchCV(pipe_ccp, param_grid_ccp, cv=5, scoring='roc_auc', n_jobs=-1)
gs_ccp.fit(X_train, yc_train)

print('Mejores hiperparámetros (con poda):')
for k, v in gs_ccp.best_params_.items():
    print(f'  {k:<35} = {v:.6f}' if isinstance(v, float) else f'  {k:<35} = {v}')

y_prob_ccp = gs_ccp.predict_proba(X_test)[:, 1]
y_pred_ccp = gs_ccp.predict(X_test)
print(f'\nCV AUC:   {gs_ccp.best_score_:.4f}')
print(f'Test AUC: {roc_auc_score(yc_test, y_prob_ccp):.4f}')
print(f'Test F1:  {f1_score(yc_test, y_pred_ccp):.4f}')

## 8 — CART para Regresión: hiperparámetros y poda

In [ ]:
param_grid_reg = {
    'tree__max_depth':        [3, 5, 7, 10, None],
    'tree__min_samples_leaf': [1, 5, 10, 20],
    'tree__ccp_alpha':        [0.0, 0.5, 1.0, 2.0, 5.0],
}

pipe_reg = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeRegressor(random_state=42)),
])

gs_reg = GridSearchCV(
    pipe_reg, param_grid_reg, cv=5,
    scoring='neg_root_mean_squared_error', n_jobs=-1, refit=True
)
gs_reg.fit(X_train, yr_train)

print('Árbol de Regresión — Mejores hiperparámetros:')
for k, v in gs_reg.best_params_.items():
    print(f'  {k:<35} = {v}')

y_pred_reg = gs_reg.predict(X_test)
rmse = mean_squared_error(yr_test, y_pred_reg) ** 0.5
print(f'\nCV RMSE:  ${-gs_reg.best_score_:.1f}')
print(f'Test RMSE: ${rmse:.1f}')
print(f'Test R²:   {r2_score(yr_test, y_pred_reg):.4f}')

# Comparar árbol óptimo vs árbol sin restricción
pipe_reg_full = Pipeline([('prep', preprocessor), ('tree', DecisionTreeRegressor(random_state=42))])
pipe_reg_full.fit(X_train, yr_train)
rmse_full = mean_squared_error(yr_test, pipe_reg_full.predict(X_test)) ** 0.5
print(f'\nÁrbol sin restricción Test RMSE: ${rmse_full:.1f} (overfitting: {rmse_full > rmse})')

## 9 — Comparación final: distintos árboles y configuraciones

In [ ]:
configs_final = [
    ('Árbol completo (sin poda)',  DecisionTreeClassifier(random_state=42)),
    ('max_depth=3',               DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('max_depth=5',               DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('min_samples_leaf=20',       DecisionTreeClassifier(min_samples_leaf=20, random_state=42)),
    ('GridSearch óptimo',         gs.best_estimator_.named_steps['tree']),
    ('GridSearch + ccp_alpha',    gs_ccp.best_estimator_.named_steps['tree']),
]

X_prep_f  = preprocessor.fit_transform(X_train)
X_prep_te = preprocessor.transform(X_test)

print(f'{"Configuración":<35} {"Depth":>6} {"Leaves":>7} {"TrainAUC":>10} {"TestAUC":>9} {"F1":>6}')
print('-' * 75)

for name, tree_model in configs_final:
    if name.startswith('GridSearch'):
        # Ya está ajustado con el preprocessor de la pipeline
        y_prob_tr = (gs if 'ccp' not in name else gs_ccp).predict_proba(X_train)[:, 1]
        y_prob_te = (gs if 'ccp' not in name else gs_ccp).predict_proba(X_test)[:, 1]
        y_pred_te = (gs if 'ccp' not in name else gs_ccp).predict(X_test)
        depth  = tree_model.get_depth()
        leaves = tree_model.get_n_leaves()
    else:
        tree_model.fit(X_prep_f, yc_train)
        y_prob_tr = tree_model.predict_proba(X_prep_f)[:, 1]
        y_prob_te = tree_model.predict_proba(X_prep_te)[:, 1]
        y_pred_te = tree_model.predict(X_prep_te)
        depth  = tree_model.get_depth()
        leaves = tree_model.get_n_leaves()

    tr_auc = roc_auc_score(yc_train, y_prob_tr)
    te_auc = roc_auc_score(yc_test,  y_prob_te)
    f1     = f1_score(yc_test, y_pred_te)
    print(f'{name:<35} {depth:>6} {leaves:>7} {tr_auc:>10.4f} {te_auc:>9.4f} {f1:>6.4f}')

## Resumen

| Hiperparámetro | Dirección para reducir overfitting | Típico punto de partida |
|---|---|---|
| `max_depth` | Reducir | 3–8 |
| `min_samples_split` | Aumentar | 10–50 |
| `min_samples_leaf` | Aumentar | 5–20 |
| `max_features` | Reducir (si hay muchas features) | `sqrt`, `log2` |
| `ccp_alpha` | Aumentar (0 = sin poda) | Buscar con CV sobre `pruning_path` |

**Flujo recomendado:**
1. Árbol sin restricción → diagnóstico de overfitting
2. Validation curve sobre `max_depth` → rango candidato
3. `GridSearchCV` con `max_depth`, `min_samples_leaf`, `ccp_alpha`
4. Verificar feature importances con `permutation_importance`

> CART es la base de los métodos ensemble: **`09_ensemble_methods.ipynb`** extiende estos árboles con Bagging, Random Forests y Gradient Boosting.

**Siguiente:** `05_model_evaluation.ipynb`